In [7]:
import sys
sys.path.append("../../")
import pandas as pd

df_stroke = pd.read_csv(r"../data/wids_pre/wids.csv")

In [8]:
from callmefair.search._search_base import CType, combine_attributes

combined_df = combine_attributes(df_stroke, cols=['neighborhood_education','neighborhood_majority_white','state_privileged'], operation=CType.intersection)
combined_df

,patient_age,breast_cancer_diagnosis_code,cancer_area,metastatic_cancer_diagnosis_code,population,female,married,income_household_150_over,income_household_six_figure,income_individual_median,rent_median,education_bachelors,education_graduate,DiagPeriodL90D,race,payer,neighborhood_affluence,neighborhood_employment,neighborhood_education_neighborhood_majority_white_state_privileged
0,0,45,21,2,31437.75000,50.142857,36.571429,7.528571,19.100000,24563.57143,1165.000000,8.357143,3.257143,1,1,0,0,0,0
1,0,27,26,0,39121.87879,50.106061,50.245455,29.596970,49.357576,41287.27273,2003.125000,23.739394,12.245455,1,1,1,1,1,0
2,0,16,3,0,21996.68333,49.876667,55.753333,18.680000,39.555000,40399.03333,1235.907407,19.678333,10.115000,1,1,1,1,1,0
3,0,20,22,0,32795.32558,50.933333,52.604762,38.057143,56.907143,55336.28571,2354.738095,33.285714,22.459524,0,1,1,1,1,0
4,0,7,21,0,10886.26000,47.688000,57.882000,8.606000,22.226000,29073.18367,919.743590,13.978000,5.684000,0,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12818,0,27,26,0,19413.05882,51.735294,36.429412,7.358824,18.352941,27888.52941,772.647059,14.400000,8.370588,1,1,1,0,0,0
12819,0,44,19,0,30153.87952,50.272840,53.076543,37.245000,55.248750,52778.65000,2223.445946,26.903704,18.277778,1,1,1,1,1,0
12820,0,44,19,2,32795.32558,50.933333,52.604762,38.057143,56.907143,55336.28571,2354.738095,33.285714,22.459524,1,1,1,1,1,0
12821,0,7,21,0,71374.13158,52.331579,39.923684,21.318421,36.207895,39491.78947,1678.447368,24.371053,16.655263,0,1,1,0,0,0


In [9]:
combined_df.to_csv('wids_education_majority_white_state_privileged.csv',index=False)

In [ ]:
from sklearn.model_selection import train_test_split
from callmefair.util.fair_util import BMInterface
from callmefair.mitigation.fair_bm import BMManager

df_train, df_test= train_test_split(combined_df, test_size=0.3, stratify=combined_df[['age_ever_married', 'stroke']] ,random_state=42)
df_test, df_val= train_test_split(df_test, test_size=0.5, stratify= df_test[['age_ever_married', 'stroke']],random_state=42)

# Defining the name of the label column
label_name = 'stroke'
# Define the name of the privileged group
sensitive_attribute = ['age_ever_married']

bm_interface = BMInterface(df_train, df_val, df_test, label_name, sensitive_attribute)

privileged_groups = [{'age_ever_married': 1}]
unprivileged_groups = [{'age_ever_married': 0}]

bm_manager = BMManager(bm_interface, privileged_groups, unprivileged_groups)

In [ ]:
from pytorch_tabnet.tab_model import TabNetClassifier
from callmefair.mitigation.fair_grid import BMGridSearch
from callmefair.mitigation.fair_bm import BMType

tabnet = TabNetClassifier(seed=42)

bm_combinations = [
    [BMType.preReweighing],  # Only preprocessing
    [BMType.preDisparate],   # Only disparate impact remover
    [BMType.preReweighing, BMType.posCalibrated],  # Preprocessing + postprocessing
]

# Initialize grid search
grid_search = BMGridSearch(
    bmI=bm_interface,
    model=tabnet,
    bm_list=bm_combinations,
    privileged_group=privileged_groups,
    unprivileged_group=unprivileged_groups
)

# Run comprehensive evaluation
grid_search.run_single_sensitive()

---

In [ ]:
df_train, df_test= train_test_split(df_stroke, test_size=0.3, stratify=df_stroke[['age', 'stroke']] ,random_state=42)
df_test, df_val= train_test_split(df_test, test_size=0.5, stratify= df_test[['age', 'stroke']],random_state=42)

# Define the name of the privileged group
sensitive_attribute = ['age']
privileged_groups = [{'age': 1}]
unprivileged_groups = [{'age': 0}]

bm_interface = BMInterface(df_train, df_val, df_test, label_name, sensitive_attribute)
bm_manager = BMManager(bm_interface, privileged_groups, unprivileged_groups)

# Initialize grid search
grid_search = BMGridSearch(
    bmI=bm_interface,
    model=tabnet,
    bm_list=bm_combinations,
    privileged_group=privileged_groups,
    unprivileged_group=unprivileged_groups
)

# Run comprehensive evaluation
grid_search.run_single_sensitive()